# Phase 6: Stage-2 ML Modelling

Train a classifier per risk factor (target = Low/Med/High code). Features = rider profile minus the factor's own defining variable(s). Models: Logistic Regression, Decision Tree, Random Forest, XGBoost. Stratified k-fold CV.

## Step 0 — Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
import joblib
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

CWD = Path.cwd()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
df     = pd.read_csv(ROOT / 'data' / 'processed' / 'model_ready.csv')
risk   = pd.read_csv(ROOT / 'data' / 'processed' / 'risk_profile.csv')
MODELS = ROOT / 'outputs' / 'models'
TABLES = ROOT / 'outputs' / 'tables'
MODELS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print('model_ready :', df.shape)
print('risk_profile:', risk.shape)

model_ready : (182, 121)
risk_profile: (182, 14)


## Step 1 — Build the analysis frame, features, and per-target exclusions

In [2]:
targets = ['force_risk_code', 'repetition_risk_code', 'duration_risk_code',
           'vibration_risk_code', 'contact_stress_risk_code', 'posture_risk_code']
data = df.copy()
for t in targets:
    data[t] = risk[t].values

# Survey feature pool used by all 6 risk factors
feature_pool = ['workload_score', 'fatigue_score',
                'age_ord', 'income_ord', 'education_ord', 'job_duration_ord',
                'work_hours_num', 'work_days_num', 'deliveries_num', 'rest_break_num',
                'vehicle_rank', 'carrying_contact_rank',
                'force_exertion', 'vibration_index',
                'workload_x_fatigue', 'workload_x_age', 'force_x_age',
                'fatigue_x_jobdur', 'deliv_x_days',
                'gender_rank', 'region_rank', 'marital_rank', 'platform_rank',
                'break_type_rank', 'tobacco_ord', 'alcohol_ord',
                'nmq_neck', 'nmq_shoulders', 'nmq_upper_back', 'nmq_lower_back',
                'nmq_elbows', 'nmq_wrists_hands', 'nmq_hips_thighs', 'nmq_knees',
                'nmq_ankles_feet',
                'd7_neck', 'd7_lower_back', 'd7_knees', 'd7_wrist_hands',
                'out_reduced_perf', 'out_taken_leave', 'out_consulted_doctor',
                'out_riding_worsens', 'out_carrying_worsens']

# Posture also takes the 11 RULA components + 8 QEC scores (the full xlsx
# observation set). Derived RULA Table A/B/C are excluded because they are
# computed FROM the 11 components and would be direct leakage on posture_risk.
rula_components = ['upper_arm', 'lower_arm', 'wrist', 'wrist_twist',
                   'neck_position', 'trunk_position', 'legs',
                   'muscle_a', 'force_a', 'muscle_b', 'force_b']

qec_scores = ['qec_back', 'qec_shoulder_arm', 'qec_wrist_hand', 'qec_neck',
              'qec_driving', 'qec_vibration', 'qec_work_pace', 'qec_stress']

exclude_per_target = {
    'force_risk_code':          ['force_exertion', 'force_x_age'],
    'repetition_risk_code':     ['deliveries_num', 'work_hours_num', 'deliv_x_days'],
    'duration_risk_code':       ['work_hours_num', 'vibration_index'],
    'vibration_risk_code':      ['vibration_index', 'vehicle_rank', 'work_hours_num'],
    'contact_stress_risk_code': ['carrying_contact_rank', 'work_hours_num'],
    'posture_risk_code':        [],
}

def features_for(target):
    base = [f for f in feature_pool if f not in exclude_per_target[target]]
    if target == 'posture_risk_code':
        return base + rula_components + qec_scores
    return base

for t in targets:
    feats = features_for(t)
    print(f'{t:28s}  -> {len(feats)} features')

force_risk_code               -> 42 features
repetition_risk_code          -> 41 features
duration_risk_code            -> 42 features
vibration_risk_code           -> 41 features
contact_stress_risk_code      -> 42 features
posture_risk_code             -> 63 features


## Step 2 — Define models and CV strategy

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, StackingClassifier)
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RNG = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)

def base_models():
    return {
        'LogReg':       LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RNG),
        'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=RNG),
        'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=RNG),
        'ExtraTrees':   ExtraTreesClassifier(class_weight='balanced', random_state=RNG),
        'HistGBM':      HistGradientBoostingClassifier(random_state=RNG),
        'XGBoost':      XGBClassifier(eval_metric='mlogloss', random_state=RNG, verbosity=0),
        'Stacking':     StackingClassifier(
            estimators=[
                ('rf',  RandomForestClassifier(class_weight='balanced', n_estimators=300, max_depth=8, random_state=RNG)),
                ('et',  ExtraTreesClassifier(class_weight='balanced',  n_estimators=300, max_depth=8, random_state=RNG)),
                ('xgb', XGBClassifier(eval_metric='mlogloss', n_estimators=300, max_depth=4, learning_rate=0.1, verbosity=0, random_state=RNG)),
                ('hgb', HistGradientBoostingClassifier(max_depth=5, learning_rate=0.1, random_state=RNG)),
            ],
            final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RNG),
            cv=3, n_jobs=-1),
    }

# Small grids per model (the Stacking ensemble is itself tuned via its components)
param_grids = {
    'LogReg':       {'clf__C': [0.1, 1.0, 10.0]},
    'DecisionTree': {'clf__max_depth': [3, 5, 7], 'clf__min_samples_leaf': [1, 3, 5]},
    'RandomForest': {'clf__n_estimators': [300, 500], 'clf__max_depth': [5, 10, None]},
    'ExtraTrees':   {'clf__n_estimators': [300, 500], 'clf__max_depth': [5, 10, None]},
    'HistGBM':      {'clf__max_depth': [3, 5, None], 'clf__learning_rate': [0.05, 0.1]},
    'XGBoost':      {'clf__n_estimators': [200, 400], 'clf__max_depth': [3, 4, 6], 'clf__learning_rate': [0.05, 0.1]},
    'Stacking':     {},
}

def smote_pipe(model, k):
    return ImbPipeline([('smote', SMOTE(random_state=RNG, k_neighbors=k)),
                        ('clf', model)])

list(base_models().keys())

['LogReg',
 'DecisionTree',
 'RandomForest',
 'ExtraTrees',
 'HistGBM',
 'XGBoost',
 'Stacking']

## Step 3 — 5-fold CV across all 5 factors x 4 models

In [4]:
from sklearn.preprocessing import LabelEncoder
from collections import Counter

scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']

cv_rows = []
best_params_log = {}
for target in targets:
    feats = features_for(target)
    X = data[feats]
    y = LabelEncoder().fit_transform(data[target])

    # SMOTE k_neighbors must be < smallest class size within each training fold
    min_class = min(Counter(y).values())
    # leave headroom for the 4/5 training fraction during 5-fold CV
    k = max(1, min(5, int(min_class * 0.8) - 1))

    for name, model in base_models().items():
        pipe = smote_pipe(model, k=k)
        gs = GridSearchCV(pipe, param_grids[name], cv=cv, scoring='f1_macro',
                          n_jobs=-1, refit=False)
        gs.fit(X, y)
        best_params_log[(target, name)] = gs.best_params_

        tuned_pipe = smote_pipe(base_models()[name], k=k)
        tuned_pipe.set_params(**gs.best_params_)
        s = cross_validate(tuned_pipe, X, y, cv=cv, scoring=scoring,
                           return_train_score=True, n_jobs=-1)
        cv_rows.append({
            'target':              target.replace('_risk_code', ''),
            'model':               name,
            'cv_accuracy':         round(s['test_accuracy'].mean(),  3),
            'cv_accuracy_std':     round(s['test_accuracy'].std(),   3),
            'cv_f1_macro':         round(s['test_f1_macro'].mean(),  3),
            'cv_precision_macro':  round(s['test_precision_macro'].mean(), 3),
            'cv_recall_macro':     round(s['test_recall_macro'].mean(),    3),
            'train_accuracy':      round(s['train_accuracy'].mean(), 3),
            'overfit_gap':         round(s['train_accuracy'].mean() - s['test_accuracy'].mean(), 3),
        })

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(TABLES / 'cv_results.csv', index=False)
cv_df

,target,model,cv_accuracy,cv_accuracy_std,cv_f1_macro,cv_precision_macro,cv_recall_macro,train_accuracy,overfit_gap
0,force,LogReg,0.550,0.034,0.474,0.485,0.487,0.706,0.156
1,force,DecisionTree,0.572,0.047,0.533,0.556,0.548,0.688,0.116
2,force,RandomForest,0.615,0.056,0.560,0.586,0.578,0.930,0.315
3,force,ExtraTrees,0.571,0.056,0.506,0.523,0.512,0.953,0.382
4,force,HistGBM,0.616,0.059,0.565,0.594,0.577,1.000,0.384
5,force,XGBoost,0.599,0.034,0.544,0.589,0.554,1.000,0.401
6,force,Stacking,0.577,0.056,0.503,0.523,0.510,1.000,0.423
7,repetition,LogReg,0.549,0.050,0.510,0.521,0.525,0.786,0.236
8,repetition,DecisionTree,0.527,0.045,0.483,0.498,0.492,0.690,0.163
9,repetition,RandomForest,0.615,0.061,0.571,0.616,0.558,1.000,0.385


## Step 4 — Pick best model per factor and save pickled fits

In [5]:
best_rows = []
for target in targets:
    short = target.replace('_risk_code', '')
    sub = cv_df[cv_df['target'] == short].sort_values('cv_f1_macro', ascending=False)
    best_name = sub.iloc[0]['model']

    feats = features_for(target)
    X = data[feats]
    y = LabelEncoder().fit_transform(data[target])

    min_class = min(Counter(y).values())
    k = max(1, min(5, int(min_class * 0.8) - 1))

    # Refit the tuned SMOTE pipeline on the full sample and save
    pipe = smote_pipe(base_models()[best_name], k=k)
    pipe.set_params(**best_params_log[(target, best_name)])
    pipe.fit(X, y)

    out_path = MODELS / f'best_{short}.pkl'
    joblib.dump({'model': pipe, 'features': feats, 'classes': sorted(set(data[target]))}, out_path)

    best_rows.append({
        'target':        short,
        'best_model':    best_name,
        'cv_accuracy':   sub.iloc[0]['cv_accuracy'],
        'cv_f1_macro':   sub.iloc[0]['cv_f1_macro'],
        'overfit_gap':   sub.iloc[0]['overfit_gap'],
        'saved_to':      str(out_path.relative_to(ROOT)),
    })

best_df = pd.DataFrame(best_rows)
best_df.to_csv(TABLES / 'best_models.csv', index=False)
best_df

,target,best_model,cv_accuracy,cv_f1_macro,overfit_gap,saved_to
0,force,HistGBM,0.616,0.565,0.384,outputs\models\best_force.pkl
1,repetition,RandomForest,0.615,0.571,0.385,outputs\models\best_repetition.pkl
2,duration,RandomForest,0.610,0.580,0.314,outputs\models\best_duration.pkl
3,vibration,ExtraTrees,0.576,0.573,0.424,outputs\models\best_vibration.pkl
4,contact_stress,RandomForest,0.599,0.589,0.401,outputs\models\best_contact_stress.pkl
5,posture,HistGBM,0.972,0.949,0.028,outputs\models\best_posture.pkl


## Step 5 — Confusion matrices and classification reports (best model per factor)

In [6]:
cm_rows = []
report_rows = []
for _, r in best_df.iterrows():
    target_code = r['target'] + '_risk_code'
    feats = features_for(target_code)
    X = data[feats]
    raw_y = data[target_code]
    y = LabelEncoder().fit_transform(raw_y)
    present_codes = sorted(set(raw_y))
    label_names = [['Low','Medium','High'][int(c)] for c in present_codes]

    min_class = min(Counter(y).values())
    k = max(1, min(5, int(min_class * 0.8) - 1))
    pipe = smote_pipe(base_models()[r['best_model']], k=k)
    pipe.set_params(**best_params_log[(target_code, r['best_model'])])
    y_pred = cross_val_predict(pipe, X, y, cv=cv, n_jobs=-1)

    cm = confusion_matrix(y, y_pred, labels=list(range(len(present_codes))))
    print(f"\n=== {r['target']} (best = {r['best_model']}) ===")
    print('Confusion matrix (rows = true, cols = pred):', ' / '.join(label_names))
    print(pd.DataFrame(cm, index=label_names, columns=label_names))

    rep = classification_report(y, y_pred, target_names=label_names,
                                output_dict=True, zero_division=0)
    for level in label_names + ['macro avg', 'weighted avg']:
        d = rep[level]
        report_rows.append({
            'target':    r['target'], 'model': r['best_model'], 'class': level,
            'precision': round(d['precision'], 3),
            'recall':    round(d['recall'],    3),
            'f1':        round(d['f1-score'],  3),
            'support':   int(d['support']),
        })

    for i, true_lbl in enumerate(label_names):
        for j, pred_lbl in enumerate(label_names):
            cm_rows.append({'target': r['target'], 'model': r['best_model'],
                            'true': true_lbl, 'predicted': pred_lbl,
                            'count': int(cm[i, j])})

pd.DataFrame(cm_rows).to_csv(TABLES / 'confusion_matrices.csv', index=False)
report_df = pd.DataFrame(report_rows)
report_df.to_csv(TABLES / 'classification_reports.csv', index=False)
report_df


=== force (best = HistGBM) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      69      14     7
Medium   20      24    13
High      5      11    19



=== repetition (best = RandomForest) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low       9      14     3
Medium    6      51    25
High      1      21    52



=== duration (best = RandomForest) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      28       6     3
Medium   12      18    26
High      8      16    65



=== vibration (best = ExtraTrees) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      42      18     7
Medium   21      38     9
High      9      13    25



=== contact_stress (best = RandomForest) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      40      17    11
Medium   16      32    10
High      7      12    37

=== posture (best = HistGBM) ===
Confusion matrix (rows = true, cols = pred): Medium / High
        Medium  High
Medium      26     3
High         2   151


,target,model,class,precision,recall,f1,support
0,force,HistGBM,Low,0.734,0.767,0.750,90
1,force,HistGBM,Medium,0.490,0.421,0.453,57
2,force,HistGBM,High,0.487,0.543,0.514,35
3,force,HistGBM,macro avg,0.570,0.577,0.572,182
4,force,HistGBM,weighted avg,0.610,0.615,0.611,182
5,repetition,RandomForest,Low,0.562,0.346,0.429,26
6,repetition,RandomForest,Medium,0.593,0.622,0.607,82
7,repetition,RandomForest,High,0.650,0.703,0.675,74
8,repetition,RandomForest,macro avg,0.602,0.557,0.570,182
9,repetition,RandomForest,weighted avg,0.612,0.615,0.609,182


## Step 6 — Phase 6 summary

In [7]:
# One row per factor: best model + headline metrics + overfit flag
summary = best_df.copy()
summary['overfit_flag'] = summary['overfit_gap'] > 0.15
summary.to_csv(TABLES / 'phase6_summary.csv', index=False)

print('Models saved to outputs/models/:')
for p in sorted(MODELS.glob('*.pkl')):
    print(' ', p.name)
print()
summary

Models saved to outputs/models/:
  best_contact_stress.pkl
  best_duration.pkl
  best_force.pkl
  best_posture.pkl
  best_repetition.pkl
  best_vibration.pkl



,target,best_model,cv_accuracy,cv_f1_macro,overfit_gap,saved_to,overfit_flag
0,force,HistGBM,0.616,0.565,0.384,outputs\models\best_force.pkl,True
1,repetition,RandomForest,0.615,0.571,0.385,outputs\models\best_repetition.pkl,True
2,duration,RandomForest,0.610,0.580,0.314,outputs\models\best_duration.pkl,True
3,vibration,ExtraTrees,0.576,0.573,0.424,outputs\models\best_vibration.pkl,True
4,contact_stress,RandomForest,0.599,0.589,0.401,outputs\models\best_contact_stress.pkl,True
5,posture,HistGBM,0.972,0.949,0.028,outputs\models\best_posture.pkl,False
